## Import libraries

In [1]:
import numpy as np
import pandas as pd

## Load misorientation tables for different sensor types

In [2]:
HH_misorientation_table=pd.read_csv("./station_table/bb_PCA_MinT_misorientation_table_20230906.csv", sep=",")
HG_misorientation_table=pd.read_csv("./station_table/acc_PCA_MinT_misorientation_table_latest_221101.csv", sep=",")
EL_misorientation_table=pd.read_csv("./station_table/sp_PCA_MinT_misorientation_table_221205.csv", sep=",")
HH_misorientation_table.head()

,station,network,start,end,median_PCA,MAD_PCA,mean_PCA,std_PCA,median_MinT,MAD_MinT,mean_MInT,std_MinT,N_PCA,N_MinT
0,AMD,KS,2015-12-07,2025-12-31,39.697022,2.274943,39.831689,3.596152,39.540601,2.101228,39.659945,3.450029,294,334
1,BAR,KS,2008-02-20,2016-12-06,-3.794371,2.004718,-3.666887,3.084675,-3.773043,1.972967,-3.567263,3.035844,527,621
2,BAR2,KS,2017-06-12,2025-12-31,-2.362052,2.369193,-2.365829,3.744047,-2.221061,2.415494,-2.175526,3.882361,260,322
3,BGD,KG,2005-01-01,2025-12-31,-0.428947,2.271991,-0.427561,3.571485,-0.472106,2.228712,-0.467359,3.560110,1014,1080
4,BGDB,KS,2016-12-08,2017-09-20,-0.933941,3.266078,-1.157030,4.823796,-0.326245,3.071146,-0.560889,5.079433,33,40


## Load table of used stations

In [3]:
station_table=pd.read_csv("./station_table/used_stations_100km.csv", sep=",")
station_table.head()

,Network,Code,Latitude,Longitude,Elevation,Sensor
0,KS,ADOA,36.5727,128.7009,169.0,HG
1,KS,AGSA,37.0917,127.8081,104.0,HG
2,KS,ASNA,36.8277,127.0123,85.0,HG
3,KS,BLGA,36.1392,127.3032,105.0,HG
4,KS,BOGA,37.0132,127.3294,119.0,HG


## Select misorientation angle corresponding to operation period

In [4]:
def find_angle(code, origin_time):
    sensor=station_table[station_table["Code"]==code].reset_index(drop=True)["Sensor"][0]
    if sensor=="HH":
        sub_table=HH_misorientation_table[HH_misorientation_table["station"]==code].reset_index(drop=True)
        for i in range(len(sub_table)):
            start, end=sub_table["start"][i], sub_table["end"][i]
            if (start<=origin_time<=end)==True:
                angle=sub_table["median_PCA"][i]
                
    elif sensor=="HG":
        sub_table=HG_misorientation_table[HG_misorientation_table["station"]==code].reset_index(drop=True)
        for i in range(len(sub_table)):
            start, end=sub_table["start"][i], sub_table["end"][i]
            if (start<=origin_time<=end)==True:
                angle=sub_table["median_PCA"][i]
                
    elif sensor=="EL":
        sub_table=EL_misorientation_table[EL_misorientation_table["station"]==code].reset_index(drop=True)
        for i in range(len(sub_table)):
            start, end=sub_table["start"][i], sub_table["end"][i]
            if (start<=origin_time<=end)==True:
                angle=sub_table["median_PCA"][i]
                
    return angle

## Make misorientation table corresponding to each event in the catalog

In [5]:
event_catalog=pd.read_csv("/home/msseo/works/19.Korea_Directivity/49.251008_Okcheon/1.HypoInv/kim1983/Okcheon.sum", sep=',')
event_catalog

,DATE TIME,SEC,LAT,LON,DEPTH,PREF,MAG,NM,NUM,GAP,...,QASR,ID-NUM,LOC,DUR,MAG1,AMP,MAG1.1,EXT,MAG,Unnamed: 23
0,2022/11/16 12:32,41.15,36.2823,127.7609,8.63,,,,55,32,...,A,200000,,,,,,,,NaN
1,2024/01/03 18:28,11.73,36.2836,127.7610,8.81,,,,69,17,...,A,200001,,,,,,,,NaN
2,2024/01/03 18:52,50.28,36.2841,127.7597,8.55,,,,35,43,...,A,200002,,,,,,,,NaN
3,2024/12/27 21:22,8.65,36.2815,127.7609,8.63,,,,68,19,...,A,200003,,,,,,,,NaN
4,2025/04/19 10:17,15.16,36.2832,127.7609,8.69,,,,50,32,...,A,200004,,,,,,,,NaN
5,2025/10/08 02:49,39.52,36.2823,127.7608,9.00,,,,77,17,...,B,200005,,,,,,,,NaN
6,2025/10/08 02:55,41.39,36.2849,127.7576,8.42,,,,12,68,...,A,200006,,,,,,,,NaN
7,2025/10/08 03:08,19.81,36.2855,127.7576,8.02,,,,10,143,...,B,200007,,,,,,,,NaN
8,2025/10/08 03:14,32.96,36.2851,127.7606,8.07,,,,10,143,...,B,200008,,,,,,,,NaN
9,2025/10/08 03:16,5.87,36.2859,127.7581,8.78,,,,23,68,...,A,200009,,,,,,,,NaN


In [6]:
for i in range(len(event_catalog)):
    print(f"Event {i+1}\n")
    datetime=event_catalog["   DATE     TIME"][i].split(" ")[0]
    year=datetime.split("/")[0]
    month=datetime.split("/")[1]
    day=datetime.split("/")[2]
    origin_time=f"{year}-{month}-{day}"
    misorientation_list=[]
    for code in np.array(station_table["Code"]):
        try:
            angle=find_angle(code, origin_time=origin_time)
        except:
            angle=-9999
            print(code)
        misorientation_list.append(angle)
        
    print(origin_time); print("\n");
    station_table["Misorientation"]=misorientation_list
    station_table.to_csv(f"./station_table/used_stations_100km_event{i+1}.csv", index=False)

Event 1

2022-11-16


Event 2

2024-01-03


Event 3

2024-01-03


Event 4

2024-12-27


Event 5

2025-04-19


Event 6

2025-10-08


Event 7

2025-10-08


Event 8

2025-10-08


Event 9

2025-10-08


Event 10

2025-10-08


Event 11

2025-10-08


Event 12

2025-10-08


